# C10-competition-craft — Practice p16 — Solution

In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 20260804

df = pd.read_csv("../data/train.csv")
FEATURES = [c for c in df.columns if c != "outcome"]
X = df[FEATURES]
y = df["outcome"].to_numpy()

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=11)),
]).fit(X, y)

probe = X.iloc[400:430]


def predict_labels_1(X_test):
    return pipe.predict(X_test)


def predict_labels_2(X_test):
    return pd.Series(pipe.predict(X_test))


def check_results(output, X_test):
    return {
        "series": isinstance(output, pd.Series),
        "length": len(output) == len(X_test),
        "index": isinstance(output, pd.Series) and output.index.equals(X_test.index),
        "vocab": set(np.asarray(output)) <= set(np.unique(y)),
    }


checks_1 = check_results(predict_labels_1(probe), probe)
checks_2 = check_results(predict_labels_2(probe), probe)
check_order = ["series", "length", "index", "vocab"]
fail_1 = next(name for name in check_order if not checks_1[name])
fail_2 = next(name for name in check_order if not checks_2[name])

head = X.iloc[:30]
head_checks = check_results(predict_labels_2(head), head)
passes_on_head = bool(all(head_checks.values()))
(fail_1, fail_2, passes_on_head)

### Diagnoses

**Submission 1.** It violates R3 because `pipe.predict` returns an `ndarray`, so a strong validation score cannot rescue the zero-scoring package. Fix: `return pd.Series(pipe.predict(X_test), index=X_test.index)`.

**Submission 2.** It violates R4 on a distinguishing index: the new Series gets `RangeIndex(0, 30)` rather than the caller's 400–429 identities. The head probe passed only because its index accidentally equaled that default; fix by supplying `index=X_test.index`.

**Submission 3.** The contract is irrelevant after violating the never-touch grading protocol by inspecting the answer-key rows. Remove that cell entirely and base every modeling choice and reported number on the frozen training-table validation carve.

**Submission 4.** This is evaluation leakage: the 150 “validation” rows were already seen by both scaler and kNN during the all-row fit, so 0.93 measures training-like recall rather than generalization. An honest estimate requires carving first, fitting the entire pipeline only on the other 450 rows, and evaluating macro-F1 once on the untouched 150.

### Answer check

In [ ]:
assert fail_1 == "series"
assert fail_2 == "index"
assert checks_1 == {"series": False, "length": True, "index": False, "vocab": True}
assert checks_2 == {"series": True, "length": True, "index": False, "vocab": True}
assert passes_on_head is True